# WAN uplink loss and latency: analysis report

One-minute loss and latency readings for every Meraki uplink in the database. The report covers data quality, latency relative to each link's baseline, level shifts, trends, outages and the latency distribution.

In [ ]:
from wanstats import analysis, charts, config, notebook

DB = None         # None reads WAN_STATS_DB from .env. For the invented demo data: 'sqlite:///demo.sqlite'
DAYS = 7          # the period of the summary table and the site charts
THEME = 'light'   # or 'dark'

store = config.open_db(DB, create=False)
a = analysis.analyse(store.load())
store.close()
print('%d links analysed, %d flagged' % (len(a.links), len(a.flagged)))

## 1. Data completeness and quality

The number of readings, links and sites, the period covered, and the share of readings that are missing, empty or lossy. A collector gap is a stretch where at least 60% of an organization's links have no readings at the same moment. It is attributed to the collector, not to the sites, and is excluded from every outage count.

In [ ]:
notebook.checks(a)

## 2. Weekly median latency relative to baseline

One row per link and one cell per week. The baseline is the lowest level a link has held for 30 days or more. Each cell is shaded by the ratio of that week's median latency to the link's own baseline.

In [ ]:
charts.fleet(a, THEME)

## 3. Links classified as Worse or Drifting

The rise above baseline is classified on one scale: **Worse** is at least 50% and 10 ms, **Drifting** is at least 10% and 3 ms. The rise is measured in two ways, as a dated level shift and as a trend inside the current level (Theil-Sen slope, Mann-Kendall p below 0.001). Both are applied to the daily median and to the daily 95th percentile, and a link takes the worse result.

In [ ]:
notebook.show(notebook.flagged(a), *[charts.flagged(a, link, THEME) for link in a.flagged])

## 4. Detected level shifts in median latency

For every day, the median of the 14 days after is compared with the median of the 14 days before. A shift is recorded when the two differ by at least 3 ms, 10%, and four times the link's day-to-day noise. Shifts in both directions are recorded. Where two or more links shift within three days of each other, the dates are shaded.

In [ ]:
notebook.show(charts.shifts(a, THEME), notebook.shifts(a))

## 5. Summary statistics by link for the reporting period

One row per link for the last `DAYS` days. **Median** and **p95** are the 50th and 95th percentiles of latency, and **Spread** is their difference. **Bad min** is the share of minutes with 2% loss or more, or latency over 150 ms. **Avail** is the share of the period outside an outage. **Latency** is the classification from section 3.

In [ ]:
notebook.summary(a, DAYS)

## 6. Outages: total downtime by link and longest events

An outage is three minutes or more of at least 80% loss, or more than ten minutes with no readings from a link while the rest of its organization kept reporting.

In [ ]:
notebook.show(charts.downtime(a, THEME), notebook.outages(a, top=6))

## 7. Dual-uplink sites with an outage in the reporting period

Hourly median latency of both uplinks, for each site with two uplinks where one had an outage of 30 minutes or more in the last `DAYS` days.

In [ ]:
sites = notebook.dual_sites(a, DAYS)
notebook.show(*[charts.site(a, name, DAYS, THEME, org=org) for org, name in sites])
if not sites:
    print('No site with two uplinks had an outage in the last %d days.' % DAYS)

## 8. Latency distribution by link

Median, interquartile range, and 5th to 95th percentile of every stored reading, one box per link, sorted by median.

In [ ]:
charts.boxes(a, THEME)

## 9. Readings of one link as a DataFrame

`a.links` holds every link with its readings, daily values, classification and outages. `notebook.frame()` returns one link's readings as a pandas DataFrame for further analysis. The example summarizes one link in 30-day periods.

In [ ]:
link = a.flagged[0] if a.flagged else a.links[0]
frame = notebook.frame(link)
print(link.label)
frame.set_index('time')['latency_ms'].resample('30D').agg(['count', 'median', 'max']).round(1)

## 10. Export to PDF or HTML

Run the cells from top to bottom, or let nbconvert run them and write the report with the code hidden:

```
jupyter nbconvert --to webpdf --no-input --execute wan-stats.ipynb
jupyter nbconvert --to html --no-input --execute wan-stats.ipynb
```